# Analista de Marketing i Estratégia Comercial: 06/07/2026
Quines característiques dels allotjaments
(equipaments, capacitat i ubicació) estan més relacionades amb els preus a cada ciutat?

## Análisis explicativo del precio — Fase 1 (Global) y Fase 2 (por ciudad)

Objetivo: cuantificar el peso de la capacidad, los equipamientos y la ciudad sobre 
el precio de los alojamientos, mediante regresión lineal múltiple (MCO) sobre log(precio).

Enfoque explicativo, no predictivo. Punto de partida: dataset ya transformado 
(flags de amenities por categoría, nulos de price/accommodates eliminados, logp calculado).

# Configuración del Entorno y Librerías
En este primer paso técnico, cargaremos las librerías necesarias para la manipulación de datos, la modelización econométrica mediante Mínimos Cuadrados Ordinarios (MCO) y la aplicación de los parches de robustez matemática para los contrastes.

* `pandas` y `numpy`: Para la carga de datos, limpieza de strings y transformaciones logarítmicas.
* `statsmodels`: Herramienta principal para ejecutar la regresión lineal múltiple y obtener los estadísticos de validación (R², p-valores y coeficientes).
* `scipy.stats`: Para las pruebas complementarias de distribución y diagnóstico de residuos.

In [15]:
from pathlib import Path
import pandas as pd

Ruta para importación del csv

# se tiene que cambiar por el clean_dataset cuando este terminado

## 0. Carga del dataset transformado

Se importa el dataset resultante de la fase de Transformation (con las categorías 
de amenities ya codificadas como flags binarios y `logp` disponible). 
Verificación rápida de dimensiones y de que las columnas esperadas (accommodates, 
las 7 categorías de amenities, city, logp) están presentes antes de modelar.

In [16]:
def encontrar_raiz_proyecto(nombre_carpeta='Equip_34'):
    '''
    Función para encontrar la carpeta raíz del proyecto subiendo desde el directorio actual.
    '''
    actual = Path.cwd()
    for carpeta in [actual] + list(actual.parents):
        if carpeta.name == nombre_carpeta:
            return carpeta
    raise FileNotFoundError(f"No se encontró la carpeta '{nombre_carpeta}' subiendo desde {actual}")

raiz_proyecto = encontrar_raiz_proyecto('Equip_34')
ruta = raiz_proyecto / 'Data' / 'raw_dataset_06_07_2026.csv'

print(f"Ruta resuelta: {ruta}")
df_original = pd.read_csv(ruta)
df = df_original.copy()

Ruta resuelta: /Users/didi/Desktop/Data_Analisis/simulador_ita/ProjecteData/Equip_34/Data/raw_dataset_06_07_2026.csv


In [ ]:
df.shape



(8000, 35)

## 1. Diagnóstico previo de los supuestos del modelo

Antes de ajustar cualquier regresión, se comprueba que los supuestos básicos de MCO 
se sostienen sobre estos datos:
- Asimetría de price vs logp (para confirmar que el log corrige el sesgo).
- Multicolinealidad entre predictoras numéricas (VIF), especialmente accommodates.
- Verificación de que no quedan duplicados de apartment_id que rompan la independencia 
  de observaciones.

Este diagnóstico se hace una sola vez, sobre el dataset global, y es válido tanto 
para la Fase 1 como para la Fase 2.

## 1.1 Tratamiento de nulos en variables críticas

Antes de ajustar cualquier modelo, se eliminan los registros con `price` o 
`accommodates` nulo (171 nulos detectados en `price`). No se imputan, al ser 
las dos variables centrales del análisis explicativo — imputar aquí introduciría 
sesgo directo en los coeficientes del modelo. Esta eliminación se hace una única 
vez sobre el dataset de trabajo, antes de la Fase 1 y la Fase 2.

## 1.2 Verificación y resolución de duplicados de apartment_id

Se confirma si existen duplicados de apartment_id en el dataset de trabajo. 
De existir, se conserva un único registro por alojamiento (último snapshot 
según insert_date), para garantizar independencia de observaciones antes 
de ajustar cualquier modelo. Este filtrado se aplica una sola vez, junto 
con el tratamiento de nulos de la sección anterior.

## 2. Fase 1 — Modelo Global (ciudad como variable de control)

Se ajusta un único modelo de regresión múltiple sobre todo el dataset:
log(precio) ~ capacidad + categorías de equipamientos + ciudad (dummies)

Aquí la ciudad se incluye como variable de control, no como segmentación: el objetivo 
es aislar el efecto neto de capacidad y equipamientos evitando que una ciudad cara 
(con alojamientos de mayor capacidad y más extras) infle artificialmente esos coeficientes.

Se usan errores estándar robustos (HC3) para corregir la posible heterocedasticidad.

## 3. Fase 1 — Lectura de resultados del modelo global

Interpretación de los coeficientes del modelo global:
- Dirección y magnitud (en % vía semi-elasticidad, al estar en log) de accommodates 
  y de cada categoría de equipamientos.
- Significatividad estadística de cada término (p-valor).
- Qué ciudades muestran un efecto significativamente distinto respecto a la ciudad 
  de referencia (dummy base).

Esta lectura responde a la pregunta "en promedio, en toda España, qué pesa más" — 
todavía no responde a "en cada ciudad", que es la Fase 2.

## 4. Fase 1 — Reparto de varianza por bloque (Eta cuadrado)

Cálculo del η² agrupado por bloque de variables (capacidad / bloque de amenities / ciudad) 
mediante ANOVA tipo II sobre el modelo global.

Esto cuantifica qué porcentaje de la variabilidad total del precio explica cada bloque 
en conjunto, complementando la lectura de coeficientes individuales de la celda anterior 
con una visión de "peso relativo" más fácil de comunicar a negocio.

## 5. Fase 2 — Modelos locales (uno por ciudad)

Se ajusta un modelo de regresión independiente para cada ciudad:
log(precio) ~ capacidad + categorías de equipamientos

Aquí la ciudad ya no es variable de control sino criterio de segmentación: cada modelo 
responde directamente a la pregunta de negocio ("¿qué características pesan más en 
CADA ciudad?"), permitiendo que el peso de capacidad y equipamientos varíe libremente 
de un mercado a otro.

Se registra, para cada ciudad: tamaño de muestra (n), R² del modelo, y coeficientes 
por variable.

## 6. Fase 2 — Comparación entre ciudades

Se construye una tabla comparativa (ciudad × variable) con el peso de capacidad 
frente al peso del bloque de equipamientos en cada mercado local.

Se presta especial atención a los casos con n bajo (Menorca, Valencia, Málaga, Sevilla), 
donde el R² y los coeficientes son menos fiables y deben leerse con más cautela 
que en ciudades con mayor volumen (Barcelona, Madrid).

## 7. Visualización — Heatmap característica × ciudad

Construcción del heatmap que resume visualmente el resultado de la Fase 2: filas = 
categorías de características (capacidad + cada bloque de amenities), columnas = 
ciudades, valores = peso/coeficiente de cada una en el modelo local correspondiente.

Es la pieza visual central del análisis, pensada para condensar en una sola figura 
los dos niveles del estudio (global vs local) para la presentación a negocio.

## 8. Validación de robustez

Chequeos finales sobre el/los modelo(s) ajustado(s):
- Test de heterocedasticidad (Breusch-Pagan) para confirmar que los errores robustos 
  (HC3) eran necesarios.
- Revisión de residuos (normalidad aproximada, ausencia de patrones).
- Confirmación de que no eliminar outliers no distorsiona los resultados principales 
  (sensibilidad del modelo con y sin los precios más extremos, solo como control, 
  no como tratamiento).

## 9. Conclusiones del análisis

Síntesis de hallazgos, respondiendo explícitamente a la pregunta de negocio: 
qué características (equipamientos, capacidad, ubicación) están más relacionadas 
con el precio, tanto a nivel global como diferenciando por ciudad.

Se documentan también las limitaciones del enfoque (asociación no causalidad, 
R² parcial acotado a las variables seleccionadas, fiabilidad limitada en ciudades 
de muestra pequeña) y las líneas de mejora futura (ampliar variables, modelo 
multinivel por ciudad).